# Ablation planning with Neural Cellular Automata
### MICCAI 2026 Tutorial · *Learning to Self-Organize* · Hands-on 3

**Jonas Mehtali** · University of Strasbourg / ICube

---

**The task.** Microwave ablation of liver tumours. The clinician pushes one to
four needles into the liver, sets a power, and needs to know **which tissue
will die**. The reference answer is a Pennes bioheat equation coupled to a
microwave SAR field and an Arrhenius damage integral, solved by
[UniPhys](https://github.com/JonasMht/UniPhys). That solve produced every label
in this notebook.

One solve takes about **20 seconds per plan**. A planner that searches over
plans needs thousands of them. This notebook trains a surrogate.

**Why an NCA suits this task.** Heat diffusion *is* a local update
rule iterated over time. Every term of the bioheat equation is a function of a
voxel and its immediate neighbours:

$$\rho c\,\frac{\partial T}{\partial t} \;=\; \underbrace{\nabla\!\cdot\!(k \nabla T)}_{\text{neighbours}}
 \;+\; \underbrace{\omega_b \rho_b c_b (T_b - T)}_{\text{perfusion}}
 \;+\; \underbrace{Q_{\text{ext}}}_{\text{the needle}}$$

An NCA is a *learned* local update rule iterated over steps. The inductive bias matches the structure of the
problem.

**What you will build.** A chained NCA of about **ten thousand parameters**. It
is given three things: a CT slice, a needle, a power. It produces two outputs:
the necrosis field, and a segmentation of the vessels it had to find to get the
necrosis right. It runs in milliseconds and sits under an interactive planner.

The vessel head matters: vessels carry heat away, so a model that cannot find
them cannot get the necrosis right near one, and the notebook never tells the
model where they are. It is also hard. On the CT the model is given, the best
single Hounsfield threshold finds the vessels with **F1 0.44**. Vessels differ
from parenchyma by shape, the branching and tapering, and a local rule
iterated over many steps can compute shape.

**Every number in this notebook was measured** on real patient CT. Ideas that
measured wrong are marked as such.

**Time budget:** 30 minutes. **Runtime → Change runtime type → T4 GPU** before
you start.

---
*Repo: https://github.com/JoKalk/colab_test_miccai2026 · branch `ablation2d` — Colab test deployment; authored upstream in `miccai2026-nca-tutorial` via `build_notebooks.py`.*

## 0 · Setup

In [ ]:
import os, sys, subprocess, pathlib, dataclasses

# This notebook ships inside the repo it needs: on the `ablation2d` branch of
# colab_test_miccai2026 (package, corpus and checkpoint all travel together).
# Colab only copies the notebook itself, so clone the branch first.
REPO   = "https://github.com/JoKalk/colab_test_miccai2026"
BRANCH = "ablation2d"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("colab_test_miccai2026").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=False)

ROOT = None
for p in ("colab_test_miccai2026", ".", "..", "../.."):
    if (pathlib.Path(p) / "ablation2d" / "channels.py").exists():
        ROOT = pathlib.Path(p).resolve()
        sys.path.insert(0, str(ROOT))
        break
if ROOT is None:
    raise RuntimeError("could not find ablation2d/: is it next to this notebook?")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("root  :", ROOT)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# Test-harness switches: scripts/test_notebook.py sets these to run the whole
# notebook on a CPU in minutes. In the session, leave them alone.
QUICK = os.environ.get("NCA_NOTEBOOK_QUICK") == "1"
LIMIT = 48 if QUICK else None

# `ablation2d` is a small package of PLUMBING: data loading, plotting, the
# physics bridge, the training loop. Everything that IS the lesson gets typed
# out in this notebook. If a cell imports something, it is not the point.
from ablation2d import viz
from ablation2d.channels import CHANNEL_NAMES
from ablation2d.data import AblationDataset
from ablation2d.anatomy import from_segmentations
from ablation2d.train import FAST

## 1 · The physics we are replacing

Every label in this notebook was solved by UniPhys on **real patient CT**: 4 000
plans on 2 000 axial slices from 50 livers, split **by patient** (35 / 5 / 10) so
that no near-duplicate slice of a test liver was ever trained on.

Three properties of the physics matter here:

* **liver** is perfused: blood carries heat away everywhere, at a rate the
  Pennes term calls $\omega_b$;
* **vessels** are a *convective sink*, $\mathrm{Nu}\,k_b/(R^2\rho c)$, switched
  off below a 1.5 mm radius. Lu et al. (AJR 2002) found heat sink in 32 of 44
  hepatic veins above 3 mm diameter and **none of 20 below**; without that floor
  the $1/R^2$ makes a venule a *better* heat sink than the portal vein, which is
  wrong;
* the needle deposits its power into a **radiating slot** 11.8 mm long, 7.25 mm
  back from the tip, not along the whole shaft: the calibrated geometry of a
  Medtronic Emprint HP, and the reason the necrosis is a sphere and not a cigar.

In [ ]:
data = {s: AblationDataset(ROOT / "data_liver" / f"{s}.npz", device=DEVICE, limit=LIMIT)
        for s in ("train", "val", "test")}
for s, d in data.items():
    print(f"{s:6s} {len(d):5d} samples   grid {d.shape}")
print()
print(data["train"].stats())

In [ ]:
# One case, end to end. The input decision with the largest measured effect is
# visible here. Left: the CT as scanned. Middle: what the model is given.
# Right: what the solver computed.
full = AblationDataset(ROOT / "data" / "test.npz", limit=8)      # unmasked, for show
i = 3
x = data["test"].inputs[i].cpu().numpy()
y = data["test"].cell_death[i, 0].cpu().numpy()
to_hu = lambda ch: ch * 2000.0 - 1000.0          # the stored channel is normalised

fig, ax = plt.subplots(1, 3, figsize=(13, 4.4), dpi=110)
needle = np.where(x[CHANNEL_NAMES.index("needle")] > 0, 1, np.nan)
viz.show_ct(to_hu(full.inputs[i, 0].numpy()), ax[0], "the CT as scanned")
viz.show_ct(to_hu(x[0]), ax[1], "what the model sees: CT masked to the liver")
for a in ax[:2]:
    a.imshow(needle, cmap="autumn", origin="lower", interpolation="nearest")
viz.show_necrosis(y, ax[2], title="UniPhys necrosis (Arrhenius d >= 0.99)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f"necrosis: {(y >= 0.99).sum() * 0.04:.1f} cm2    "
      f"CT kept on {100 * (x[0] > 0).mean():.0f} % of the slice")

### Three channels in, two out, and why the CT is masked

| # | in | what it is |
|---|----|-----------|
| 0 | `hounsfield` | the CT slice, zeroed outside the liver |
| 1 | `needle` | the applicator shaft |
| 2 | `applicator_activation` | delivered power at the radiating slot |

| # | out | what it must produce |
|---|-----|---------------------|
| 0 | `cell_death` | the Arrhenius damage field |
| 1 | `vessel` | where the vessels are |

**Why mask the CT.** Three quarters of every slice is not liver: bowel, spine,
ribs, air, and all of it is a distractor for a model with ten thousand
parameters. Masking it is one of the largest effects measured on this corpus,
and it replicated at two different loss weights:

| | necrosis DSC | vessel F1 | best HU threshold, same input |
|---|---|---|---|
| full CT | 0.848 | 0.611 | 0.245 |
| **CT masked to the liver** | **0.868** | **0.749** | 0.439 |

It helps *both* heads. In the last column, masking helps the plain threshold
more than it helps the model. A baseline must therefore be measured on **the
same input the model sees**. Against the full-CT threshold this model looks
three times better than a `>`; against the same-input threshold it is 1.7×.

The mask has one checkable precondition: every vessel voxel in this corpus
lies *inside* the liver mask, so the mask's outline carries no information
about where the vessels are. If a vessel sat outside the parenchyma, the mask
outline would reveal its location. (`tests/test_smoke.py` pins that.)

It is also a real assumption about deployment: the model needs a liver contour
at inference. An ablation planning system has one.

**Burn time is fixed at 5 minutes** and is therefore not an input. A corpus that
varied duration with no duration channel would ask the model to predict the
average over every duration it might have had, a target that is not a function
of the inputs.

**An earlier design gave the model the physics instead**: eight channels of
$\rho c$, perfusion, $k$, $\sigma$ and a vessel sink rate. On a *procedural*
corpus it scored DSC 0.93, but it assumed someone had already segmented the
tissue and looked up its properties, which is itself a hard task. A planning
system has a CT.

In [ ]:
fig, _ = viz.show_channels(x)
plt.show()

### One modelling assumption

The label is **solved in 3-D and then sliced**. The anatomy is extruded along
$z$, the needles lie in the mid-plane, and the label is the mid-plane of the full
3-D solve.

Why not solve in 2-D? Because a one-cell-deep grid is the physics of an
*infinite slab*: no heat leaves through the faces. Measured on this grid, 90 W
for 5 min: the 2-D slab gives a **40 × 48 mm** necrosis where the mid-plane of
the same needle in 3-D gives **40 × 42 mm**: same width, 14 % too long. The depth
is what makes the target a deterministic function of the slice the model is
shown, which is the difference between a learnable problem and a noisy one.

## 2 · The model

A **chained** NCA (C-NCA). One CA step is not one convolution block but $N$ of
them in sequence, each adding a residual:

```
for sub in sub_models:      grid = grid + sub(grid)
```

The whole state is one tensor, and its channels have jobs:

| channels | role | written by the model? |
|---|---|---|
| 0 - 1 | the **answers**: necrosis, vessels | yes |
| 2 - 4 | the **environment**: CT, needle, power | no, rewritten from the inputs every step |
| 5 + | **scratch** channels to think in | yes |

**Reach.** Each sub-model is two 3×3 convolutions, so one step moves
information $2N$ cells. It has to cover the distance from the radiating slot to
the edge of a necrosis, about a dozen cells. Get this wrong and the failure is
silent: the cells at the edge receive no information about the needle. The
model below prints its own reach.

**Scratch channels are not optional.** With 2 answers and 3 environment channels
pinned, a 6-channel model has **one** free channel and scores necrosis DSC 0.30.
At 16 channels, eleven free, it scores 0.71 and the vessel head rises with it.
Loss tuning did not move these numbers; adding channels did.

In [ ]:
N_OUT = 2          # channels 0-1: necrosis, vessels
N_INPUT = 3        # channels 2-4: CT, needle, applicator activation
OUT = slice(0, N_OUT)
COND = slice(N_OUT, N_OUT + N_INPUT)
STATE_CLAMP = 4.0

In [ ]:
class AblationCNCA(nn.Module):
    def __init__(self, channels=12, hidden_mult=2, n_sub_models=2, fire_rate=0.5):
        super().__init__()
        self.channels, self.fire_rate = channels, fire_rate
        self.n_sub_models = n_sub_models
        self.restore_conditioning = True        # the playground switches this off
        hidden = channels * hidden_mult
        self.sub_models = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, hidden, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(hidden, channels, 3, padding=1, bias=False),
            ) for _ in range(n_sub_models)
        ])
        for sub in self.sub_models:
            fan_in = sub[0].weight[0].numel()
            nn.init.normal_(sub[0].weight, 0.0, (2.0 / fan_in) ** 0.5 * 0.1)
            nn.init.zeros_(sub[0].bias)
            nn.init.zeros_(sub[2].weight)       # every sub-model starts as a no-op

    def step(self, x):
        for sub in self.sub_models:
            dx = sub(x)
            mask = (torch.rand_like(x[:, :1]) <= self.fire_rate).to(x.dtype)
            x = STATE_CLAMP * torch.tanh((x + dx * mask) / STATE_CLAMP)
        return x

    def seed(self, inputs):
        b, _, h, w = inputs.shape
        x = inputs.new_zeros(b, self.channels, h, w)
        x[:, COND] = inputs
        return x

    def forward(self, inputs, steps=15, state=None, return_trace=False,
                return_state=False, trace_full=False):
        x = self.seed(inputs) if state is None else state
        trace = []
        for _ in range(steps):
            x = self.step(x)
            if self.restore_conditioning:
                x = torch.cat([x[:, OUT], inputs, x[:, COND.stop:]], dim=1)
            if return_trace:
                frame = x if trace_full else torch.sigmoid(x[:, OUT])
                trace.append(frame.detach().clone())
        out = torch.sigmoid(x[:, OUT])
        if return_trace:
            return (out, trace, x) if return_state else (out, trace)
        return (out, x) if return_state else out


model = AblationCNCA(channels=FAST.channels, hidden_mult=FAST.hidden_mult,
                     n_sub_models=FAST.n_sub_models).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
scratch = model.channels - N_OUT - N_INPUT
print(f"{n_params:,} parameters · {model.channels} channels = "
      f"{N_OUT} answers + {N_INPUT} environment + {scratch} scratch")
print(f"reach: {2 * model.n_sub_models} cells per step")

**About ten thousand parameters.** A 3-D U-Net surrogate for the same task is
in the tens of millions: three orders of magnitude larger.

### The free sanity check

Every sub-model's output convolution is zero-initialised, so an untrained model
must produce an **exactly flat** field: not nearly flat, flat. It takes two
seconds and it catches a wrong state layout, a wrong slice or a wrong readout
before training.

In [ ]:
with torch.no_grad():
    p = model(data["val"].inputs[:1], steps=20)
spread = float(p.max() - p.min())
print(f"output shape {tuple(p.shape)}   untrained spread = {spread:.3e}")
print("FLAT, good" if spread == 0.0 else "NOT FLAT: the bug is in step() or seed()")

### The loss: two heads, two losses, one balance

**Necrosis** is a continuous damage field with a meaningful interior, so it gets
a *weighted* mean squared error. Necrosis covers 1-4 % of the slice, and an
unweighted MSE is minimised by predicting zero everywhere, so foreground is
weighted **100 : 1**. It says a missed necrotic cell costs a hundred invented
ones. For an ablation margin, under-prediction is the safer error, and the
ratio is a clinical choice, not a neutral default.

**Vessels** are a thin binary mask covering about 3 % of the slice, so they get
**Dice + weighted binary cross-entropy**.

The two are added: `L = L_necrosis + w · L_vessel`. **That `w` turned out to be
one of the most consequential numbers in this notebook**, and you will see why after
training. First, write the necrosis half yourself.

In [ ]:
def necrosis_loss(pred, target, fg=100.0, bg=1.0, thr=0.01):
    w = torch.where(target > thr, torch.full_like(target, fg), torch.full_like(target, bg))
    return (w * (pred - target) ** 2).sum() / w.sum()


@torch.no_grad()
def dice(pred, target, contour=0.99, thr=0.5):
    p = (pred > thr).flatten(1).float()
    t = (target >= contour).flatten(1).float()
    inter = (p * t).sum(1)
    den = p.sum(1) + t.sum(1)
    return torch.where(den > 0, 2 * inter / den, torch.ones_like(den))

In [ ]:
# Check your necrosis_loss on two cases small enough to do in your head.
p_ = torch.tensor([[[[0.0, 1.0]]]])
t_ = torch.tensor([[[[1.0, 1.0]]]])     # both foreground, weight 100 each; errors 1, 0
assert abs(float(necrosis_loss(p_, t_)) - 0.5) < 1e-6, float(necrosis_loss(p_, t_))
t_ = torch.tensor([[[[1.0, 0.0]]]])     # weights 100 and 1; errors 1 and 1
assert abs(float(necrosis_loss(p_, t_)) - 1.0) < 1e-6, float(necrosis_loss(p_, t_))
print("necrosis_loss is correct")

## 3 · Training

The training loop is plumbing, so it is imported. The tricks inside it are
the lecture, and each is flagged `# TRICK n` in `ablation2d/train.py` so the
slides and the code point at the same lines:

1. **Sample the rollout length** every batch. A fixed K fits one trajectory
   length; run one step longer and the output degrades.
2. **Start from states the model actually reaches**, not only from a blank grid.
3. **Sometimes seed the state with the target** and train it to stay there.
   Switched *off* in this recipe (`target_to_seed_rate = 0`); the switch stays
   in the code.
4. **Normalise gradients per parameter**, then clip as a second net.
5. **Persistence** with truncated back-propagation: supervise a second, longer
   stretch of the *same* rollout, so the field holds when you keep iterating.
   The planner's step slider depends on it.

And one that lives in the model you wrote rather than in the loop: **stochastic
firing**. Each cell updates with probability ½, so updates are asynchronous
across the grid. One consequence: inference is stochastic, and one rollout is
one draw.

### Two ideas that measured wrong

**Data augmentation.** Rotations and flips are exact on a square grid, every
channel is a scalar field, and the bioheat equation has no preferred direction,
so eight orientations appear free. Tested as a one-variable comparison, same
data and same budget, one flag:
necrosis did not move (DSC 0.831 both ways) and the vessel head went
**F1 0.70 to 0.28**. Every scan here is an axial slice in a standard patient
orientation, the liver on the patient's right and the spine behind, and
training with invariance to that orientation removes that information from
the input. It is implemented below, correctly, and switched **off**.

**A vessel-only warm start.** Vessel masks need no solver, so pretraining the
vessel head alone first appears free. Measured: per gradient step it learned
vessels at roughly *half* the rate of training both heads together. The
necrosis task is a useful auxiliary for vessels, not a competitor.

### The loss share

With `w = 3.0` the necrosis head received **3 % of the loss**. Dropping it to
`w = 0.3`, same model, same data, one number, moved validation DSC at epoch 10
from **0.46 to 0.79**. The weight looked reasonable; the *share* was not.

In [ ]:
# The augmentation, correctly implemented, then left off (see above).
# Eight orientations; composing a further mirror lands back inside the eight.
from ablation2d.train import augment_d4, d4, N_D4

t = torch.arange(16.).reshape(1, 1, 4, 4)            # no symmetry of its own
orbit = {tuple(d4(t, g).flatten().tolist()) for g in range(N_D4)}
closed = {tuple(torch.flip(d4(t, g), dims=(-1, -2)).flatten().tolist())
          for g in range(N_D4)} == orbit
print(f"{len(orbit)} distinct orientations · closed under a further mirror: {closed}")

xb, yb = data["train"].sample(slice(0, 8))
xa, ya = augment_d4(xb, yb, torch.Generator(device=xb.device).manual_seed(0))
fig, ax = plt.subplots(2, 8, figsize=(16, 4.2), dpi=100)
for k in range(8):
    ax[0, k].imshow(to_hu(xa[k, 0].cpu().numpy()), cmap="gray", vmin=-160, vmax=240,
                    origin="lower")                    # liver window, not auto-scaled
    ax[1, k].imshow(ya[k, 0].cpu(), cmap="inferno", origin="lower")
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
ax[0, 0].set_ylabel("CT", fontsize=8); ax[1, 0].set_ylabel("necrosis", fontsize=8)
fig.suptitle("each sample gets its own orientation, and the target turns with it",
             fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
from ablation2d.train import train, choose_inference_steps
from ablation2d.live import LiveDashboard

cfg = dataclasses.replace(
    FAST, device=DEVICE,
    augment_d4=False,       # measured: vessel F1 0.70 -> 0.28, one variable
    pretrain_epochs=0,      # measured: slower than training both heads together
    vessel_weight=0.3,      # measured: 3.0 gave necrosis 3 % of the loss
    epochs=24,              # cap for fast machines
    validate_every=2,       # more dashboard updates during the session
    time_budget_s=8 * 60,   # the session budget is TIME, not epochs
)
if QUICK:
    cfg = dataclasses.replace(cfg, epochs=2, validate_every=1, time_budget_s=None)
print(f"{cfg.channels} channels · {cfg.n_sub_models} sub-models · "
      f"{cfg.epochs} epochs or {cfg.time_budget_s and cfg.time_budget_s // 60} min, "
      f"whichever comes first · vessel_weight {cfg.vessel_weight} · "
      f"augmentation {cfg.augment_d4}")

### The session budget is time, not epochs

Training runs next under an **8-minute budget**: the loop stops between epochs
when the clock runs out and keeps the best validation weights, so the same
recipe works on any GPU. On CPU, set `LOAD_PRETRAINED = True` below
before running the training cell: it loads the shipped model instead, 16
channels and 120 epochs on this corpus, test DSC 0.874, vessel F1 0.788.
Everything after this point works with either.

What to expect from the 8 minutes, measured on two machines: a workstation GPU
finishes all 24 epochs in about 6 minutes at val DSC 0.82, and a thermally
throttled laptop GPU is stopped by the budget at about the same epoch, also val
DSC 0.82. The budget equalises machines. The full 40-epoch run of this recipe
measures test DSC 0.831 with vessel F1 0.70, and the vessel head keeps improving
after that: the vessel head needs more epochs than necrosis.

In [ ]:
LOAD_PRETRAINED = False       # True: skip training, load the shipped model
if os.environ.get("NCA_NOTEBOOK_PRETRAINED") == "1":      # test harness only
    LOAD_PRETRAINED = True

In [ ]:
if LOAD_PRETRAINED:
    from ablation2d.train import load_checkpoint
    model, cfg, ck = load_checkpoint(ROOT / "checkpoints" / "ablation_cnca_liver.pt",
                                     device=DEVICE)
    cfg = dataclasses.replace(cfg, device=DEVICE)
    print(f"loaded the shipped model: {cfg.channels} channels, "
          f"best validation DSC {ck['best_val_dice']:.4f} at epoch {ck['best_epoch']}")
else:
    dash = LiveDashboard(data["val"], every=2, steps=cfg.infer_steps)
    # `model=` is the class YOU wrote. The loop is imported because it is
    # plumbing; the model is not.
    model, history = train(cfg, data["train"], data["val"], model=model, on_epoch=dash)

## 4 · The rollout, step by step

No step sees the whole grid. The necrosis expands outward from the radiating
slot, one local update at a time.

In [ ]:
with torch.no_grad():
    _, trace = model(data["test"].inputs[:1], steps=cfg.infer_steps * 2,
                     return_trace=True)
fig, _ = viz.show_rollout(trace, every=2, max_frames=8)
plt.show()

## 5 · How good is it, and against what?

DSC alone hides two clinically different failures, so this section also reports:

* **area error in cm²**: a planner sizes a margin off it;
* **recall and precision, separately**: under-treatment and over-treatment are
  not the same mistake.

The relevant baseline for ablation is not "predict zero". It is the **sphere
a clinician draws from the device chart**, sized from power and time with the
anatomy ignored. A model that cannot beat it has learned nothing about the
anatomy.

The number of inference steps is a free hyperparameter. Training does not fix
it.

In [ ]:
K, scores = choose_inference_steps(
    model, data["val"], candidates=range(4, 21, 4) if QUICK else range(4, 41, 2))
print("validation DSC by rollout length:", {k: round(v, 3) for k, v in scores.items()})
print(f"chosen K = {K}  (the cheapest within 1 % of the best)")
cfg = dataclasses.replace(cfg, infer_steps=K)

In [ ]:
from ablation2d.evaluate import fit_sphere_baseline, predict_all, report, sphere_baseline
from ablation2d.model import best_vessel_threshold

# Fit the baseline FIRST, on TRAIN. An unfitted baseline is not a comparison.
sphere = fit_sphere_baseline(data["train"])
print(f"device-chart law: r = {sphere['C']:.3f} * P^{sphere['ALPHA']:.3f} * t^{sphere['BETA']:.3f}")

# The vessel decision threshold is chosen on VALIDATION, like K. A fixed 0.5
# measures calibration, not segmentation.
with torch.no_grad():                 # in batches: a whole split at once can fill a T4
    vthr, _ = best_vessel_threshold(predict_all(model, data["val"], cfg.infer_steps),
                                    data["val"].targets)
print(f"vessel threshold {vthr:.2f}")
print()

one = report(model, data["test"], steps=cfg.infer_steps, vessel_threshold=vthr,
             baselines={"device-chart sphere": lambda x: sphere_baseline(x, sphere)})
eight = report(model, data["test"], steps=cfg.infer_steps, vessel_threshold=vthr,
               average="repeats")
rows = {"C-NCA, 1 rollout": one["C-NCA"],
        "C-NCA, 8 rollouts averaged": eight["C-NCA"],
        **{k: v for k, v in one.items() if k != "C-NCA"}}
for name, r in rows.items():
    if "dice" not in r:
        print(f"{name:28s} vessel F1 {r['vessel_f1']:.4f}")
        continue
    line = (f"{name:28s} DSC {r['dice']:.4f}  area err {r['area_err_cm2']:+.2f} cm2  "
            f"recall {r['recall']:.3f}  precision {r['precision']:.3f}")
    if "vessel_f1" in r:
        line += f"  |  vessel F1 {r['vessel_f1']:.4f}"
    print(line)

**Why two rows for the same model?** Stochastic firing means every rollout
draws a fresh random update mask, so a single rollout is one *draw* of what the
model predicts, not the prediction itself. Averaging eight is the standard
reduction. On the shipped model (300 test cases, threshold re-tuned per row) it
is worth +0.007 DSC on necrosis but +0.026 F1 on vessels: the thin vessel mask
is where one noisy draw hurts most. For a stochastic model, the answer is
defined as the mean over draws.

In [ ]:
i = 7
with torch.no_grad():
    pred = model(data["test"].inputs[i:i + 1], steps=cfg.infer_steps)[0, 0].cpu().numpy()
truth = data["test"].cell_death[i, 0].cpu().numpy()
fig, _ = viz.compare(truth, pred)
plt.show()

### Check the share, not the weight

The loss is `L_necrosis + w · L_vessel`. Below are the two terms on the model you
just trained, and the fraction of the total the necrosis head actually receives
at three weights. The terms live on very different scales: a weight that looks modest can give
one head almost nothing.

In [ ]:
from ablation2d.model import soft_dice_loss, masked_bce

xb, yb = data["val"].sample(slice(0, 16))
with torch.no_grad():
    p = model(xb, steps=cfg.infer_steps)
l_nec = float(necrosis_loss(p[:, 0:1], yb[:, 0:1]))
l_ves = float(soft_dice_loss(p[:, 1:2], yb[:, 1:2]) + masked_bce(p[:, 1:2], yb[:, 1:2], 20.0))
print(f"necrosis term {l_nec:.4f}   vessel term {l_ves:.4f}")
print()
for w in (3.0, 1.0, 0.3):
    share = l_nec / (l_nec + w * l_ves)
    print(f"w = {w:3.1f}   necrosis receives {100 * share:5.1f} % of the loss")

### The other head: where are the vessels?

The model was never given a vessel segmentation. It was given a CT, a needle and
a power, and asked for the necrosis **and** the vessels. The heat sink is
where the vessels are, so a model that cannot find them cannot notch the
necrosis in the right place. The second head makes the model's vessel estimate
inspectable, which is how one checks that the necrosis is right for the correct
reason.

Its competition is a single Hounsfield threshold, fitted on the split it is
scored on. Contrast-enhanced vessels *are* bright, and `hu > t` is a working
segmenter.

Read the error panel, not only the F1: **green** is vessel found, **blue** is vessel
missed, **red** is vessel invented.

In [ ]:
from ablation2d.model import vessel_f1
from ablation2d.evaluate import hu_threshold_vessel_f1, predict_all

with torch.no_grad():
    pv = predict_all(model, data["val"], cfg.infer_steps)     # one rollout, in batches
    pt = predict_all(model, data["test"], cfg.infer_steps)
thr, _ = best_vessel_threshold(pv, data["val"].targets)         # chosen on VAL
f1 = vessel_f1(pt, data["test"].targets, thr).mean().item()
print(f"vessel head   F1 {f1:.4f}  at threshold {thr:.2f}")
print(f"HU threshold  F1 {hu_threshold_vessel_f1(data['test']):.4f}  <- the number to beat")

In [ ]:
per = vessel_f1(pt, data["test"].targets, thr)
order = torch.argsort(per, descending=True)
pick = [order[0], order[len(order) // 2], order[-1]]          # best, median, worst

fig, axes = plt.subplots(3, 3, figsize=(10.5, 10.5), dpi=110)
for r, k in enumerate(pick):
    k = int(k)
    hu = to_hu(data["test"].inputs[k, 0].cpu().numpy())
    truth = (data["test"].targets[k, 1] > 0.5).cpu().numpy()
    got = (pt[k, 1] > thr).cpu().numpy()

    viz.show_ct(hu, axes[r, 0], "CT (masked)" if r == 0 else None)
    axes[r, 0].set_ylabel(("best", "median", "worst")[r], fontsize=10, fontweight="bold")
    viz.show_ct(hu, axes[r, 1], "vessel head" if r == 0 else None)
    axes[r, 1].imshow(np.where(got, 1, np.nan), cmap="cool", alpha=0.6,
                      origin="lower", interpolation="nearest")
    axes[r, 1].contour(truth, levels=[0.5], colors=["#ffb300"], linewidths=0.9,
                       origin="lower")

    err = np.zeros(hu.shape + (4,))
    err[got & truth] = (0.1, 0.8, 0.2, 0.9)       # found
    err[~got & truth] = (0.2, 0.4, 1.0, 0.9)      # missed
    err[got & ~truth] = (1.0, 0.2, 0.2, 0.9)      # invented
    viz.show_ct(hu, axes[r, 2], "found / missed / invented" if r == 0 else None)
    axes[r, 2].imshow(err, origin="lower", interpolation="nearest")
    axes[r, 2].set_xlabel(f"F1 {float(per[k]):.3f}", fontsize=8)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**What made the vessel head work**: three changes, in the order they were
tested. None is a loss change:

1. **Real CT instead of procedural anatomy.** On synthetic slices the vessel head
   collapsed onto the organ: precision 0.087, IoU 0.77 against the *liver*. It
   The behaviour was correct given the data: synthetic vessels were bright
   blobs, "bright blob" was the whole rule, and the model found it. Real
   vessels branch and taper, which is the shape worth learning.
2. **Scratch channels.** 6 → 16 channels lifted vessel F1 0.32 → 0.66 and necrosis
   DSC 0.30 → 0.71 *together*: two tasks competing for scratch space, not for
   gradient.
3. **Masking the CT to the liver**: +0.14 vessel F1, replicated.

The model still has a clear weakness, visible in the worst row above: it
finds the large vessels reliably and the fine intrahepatic branches poorly.

## 6 · The heat sink

A vessel crossing the ablation zone carries heat away and notches the necrosis.
Clinically this is the difference between a complete ablation and a local
recurrence, and it is the one effect a device chart cannot capture.

Below: the same plan on the demo slice **with** its vessels, and **with them
removed**. The vessel label becomes parenchyma, its CT is inpainted from the
surrounding liver, and nothing else changes. Removing a vessel from a CT means
inventing tissue where it was, so this is a demonstration, not a measurement.

Measured over 183 tip placements beside the thickest vessel on 14 test
slices: with the vessel present the predicted necrosis is **smaller in 93 %**
of them, by 0.86 cm² on average, and 4 % go the other way. The heat sink is
learned.

One detail keeps this demo honest. The model is stochastic, so two rollouts
differ even on identical input, and a search over 36 angles for the *biggest*
with/without difference can select a noise fluctuation: on this slice, with
independent seeds, it reported a *larger* necrosis with the vessel. Both
rollouts below share one seed (common random numbers), so the only difference
between them is the vessel.

For scale, here is what the *solver* itself does in a controlled version of the
same experiment: uniform liver, one straight vein of set calibre, 100 W for 6 min,
10 mm from the tip.

| vessel calibre | none | 2 mm | 3 mm | 4 mm | 6 mm | 9 mm | 13 mm |
|---|---|---|---|---|---|---|---|
| necrosis | 9.9 cm² | 8.2 | 8.1 | 7.6 | 7.6 | 7.2 | 6.5 |

Note the 2 mm column: the convective sink is switched off there, and the necrosis
still shrinks 17 %. That is the blood *material*, high perfusion and different
electrical properties, not the sink. Two mechanisms, and only one of them is
convection.

In [ ]:
from ablation2d.experiments import heat_sink_pair

# The demo runs on a REAL slice: de-identified CT and segmentation masks.
real = np.load(ROOT / "data" / "real_slice.npz")
anat = from_segmentations(real["seg_liver"], real["seg_vessel"],
                          hounsfield=real["hounsfield"])
print(viz.vessel_legend_text(anat))

# mask_ct=True: this model was trained on the masked corpus, so it must be fed
# the CT the same way. Hand it a full abdominal CT and you are testing a
# distribution it never saw, not the model.
fig = heat_sink_pair(model, anat=anat, steps=cfg.infer_steps, device=DEVICE,
                     mask_ct=True)
plt.show()

## 7 · The interactive planner

Place needles and set their power; the necrosis field follows. Every move is a
full NCA rollout from the current plan.

It shows the model the way this notebook recommends using it: eight rollouts
averaged, vessels restricted to the liver. It prints that on every render, along
with the real latency. If UniPhys is installed (it is not, on Colab), **check
against physics** solves the same plan for real and reports the DSC.

In [ ]:
from ablation2d.ui import AblationStudio

try:                                         # works only where UniPhys is installed
    from ablation2d.physics import simulate
    oracle = lambda a, p: simulate(a, p).cell_death
except ImportError:
    oracle = None

studio = AblationStudio(
    model, anat, steps=cfg.infer_steps, torch_device=DEVICE, oracle=oracle,
    mask_ct=True,           # as trained
    average="repeats",      # the model is stochastic: one rollout is one sample
    mask_vessels=True,      # a third of vessel false positives lie outside the liver
)
studio.show()

### The state, one channel at a time

The answer is two channels wide. The state is wider, and the remaining
channels are where the computation happens.

Pick a channel, scrub the step slider, and look for three things:

* **the environment channels do not move.** CT, needle and power are rewritten
  from the inputs after every step, and the widget prints their drift so you can
  see it is exactly zero.
* **the answers are drawn as logits, not probabilities.** A head sitting at ±4 is
  pinned against the clamp and has stopped learning; a sigmoid would paint that
  as a confident 0.98. The widget says `saturated` when it happens.
* **the activity curve answers "how many steps?"**: the mean per-step change
  in that channel. Where it flattens, the channel has converged.

In [ ]:
from ablation2d.ui import ChannelScope

scope = studio.scope(steps=24)          # the plan the studio is currently showing
scope.show()

In [ ]:
# Not every channel is worth a slider: rank them by how much they move.
tr = scope.rollout()                    # (steps, channels, H, W)
activity = np.abs(np.diff(tr, axis=0)).mean(axis=(0, 2, 3))
for k in np.argsort(-activity):
    print(f"{activity[k]:9.4f}   channel {k:2d}  {scope.roles[k]}")

## 8 · Limits

Three experiments. Predict the answer to each one *before* you run it.

In [ ]:
# A. Run it far past its training length. Persistence training is supposed to
#    make the field hold; find where it stops holding.
with torch.no_grad():
    for k in (cfg.infer_steps, 30, 60, 150, 400):
        p = model(data["test"].inputs[:8], steps=k)
        d = dice(p[:, 0:1], data["test"].cell_death[:8]).mean()
        print(f"K = {k:4d}   DSC {float(d):.4f}   mean necrosis {float(p[:, 0].mean()):.4f}")

In [ ]:
# B. A power the corpus never saw. The dial was sampled 30-150 W; ask for 250 W.
#    An NCA extrapolates badly.
from ablation2d.experiments import power_sweep
fig = power_sweep(model, anat, steps=cfg.infer_steps, device=DEVICE,
                  dial_w=(0, 40, 75, 110, 150, 250), mask_ct=True)
plt.show()

In [ ]:
# C. Walk the tip OUTSIDE the liver. The corpus only placed tips inside the
#    organ, and with the CT masked, the model now cannot see the
#    tissue it is burning out there. What does it predict?
from ablation2d.experiments import outside_the_liver
fig = outside_the_liver(model, anat, steps=cfg.infer_steps, device=DEVICE, mask_ct=True)
plt.show()

## 9 · What to take away

* **Heat diffusion is a local update rule.** A ten-thousand-parameter NCA can
  stand in for a 20-second PDE solve because of it.
* **Free channels are where the model computes.** Answers and environment channels are pinned;
  what is left over is the only place it can compute. One free channel was not
  enough.
* **Check the share, not the weight.** A multi-task loss weight of 3.0 looked
  reasonable and gave one head 3 % of the loss. It was one of the largest
  effects in this project, and it was a hyperparameter nobody revisited.
* **Show the model what matters.** Masking the CT to the liver improved both
  heads; three quarters of the slice was distractor.
* **A stochastic model's answer is an average.** One rollout is one draw.
* **Measure the tricks.** Augmentation and a warm start both measured wrong.
* **A surrogate inherits its ground truth's status.** Every label here comes from
  a device calibration that is **PROVISIONAL**: fitted on ex vivo bovine liver
  at 17 °C, held-out RMS 9.9 %, not clinically validated. The NCA cannot be
  better than that.

### Take it further

* Swap the target for **peak temperature** and read the 50 °C isotherm instead.
* Condition on a **tumour mask** and report the ablation margin.
* Put the surrogate in a loop and **optimise the plan**: that is what the 3-D
  version does.
* Read: Mordvintsev et al., *Growing NCA* (2020) · Kalkhof et al., *Med-NCA*
  (2023) · Mehtali et al., *C-NCA* (MICCAI 2025) · Lu et al., *AJR* 2002 ·
  Pennes (1948).

Slides, notebooks and solutions: https://github.com/JoKalk/colab_test_miccai2026

*Questions → the final Q&A, or find me at the coffee break.*